# Understanding Dataset Writing / データセット書き込みの理解

`DatasetWriter` saves rendered character images and metadata records to disk.
This notebook covers the full workflow: image I/O, JSONL streaming, and Parquet conversion.

In [ ]:
import json
from pathlib import Path
from PIL import Image
from font2dataset.renderer import FontRenderer, RenderConfig
from font2dataset.writer import WriterConfig, DatasetWriter

# Create a test output directory
test_output_dir = Path("./test_output")
print(f"Output directory: {test_output_dir}")
print("Ready")

## 1. Basic Write / 基本的な書き込み

Render a character and save it via `DatasetWriter`.

In [ ]:
# Render a character
renderer = FontRenderer("./fonts/Aclonica-Regular.ttf")
image = renderer.render("A")

# Write it
config = WriterConfig(output_dir=test_output_dir)
with DatasetWriter(config) as writer:
    filename = writer.write("A", image, "./fonts/Aclonica-Regular.ttf", index=0)
    print(f"Saved: {filename}")

# Check output structure
images_dir = test_output_dir / "images"
print(f"\nImages directory contents: {list(images_dir.glob('*.png'))}")

jsonl_path = test_output_dir / "metadata.jsonl"
print(f"JSONL exists: {jsonl_path.exists()}")
with open(jsonl_path) as f:
    for line in f:
        print(f"Record: {json.loads(line)}")

## 2. Batch Writing / バッチ書き込み

Write multiple characters from multiple fonts.

In [ ]:
from font2dataset.charset import build_charset

# Fresh output dir for this example
batch_output = Path("./test_batch_output")
batch_output.mkdir(exist_ok=True, parents=True)

fonts = ["./fonts/Aclonica-Regular.ttf", "./fonts/CherryCreamSoda-Regular.ttf"]
chars = ["A", "B", "C"]

config = WriterConfig(output_dir=batch_output)
with DatasetWriter(config) as writer:
    record_count = 0
    for font_path in fonts:
        renderer = FontRenderer(font_path)
        for idx, char in enumerate(chars):
            image = renderer.render(char)
            if image:
                writer.write(char, image, font_path, index=idx)
                record_count += 1

print(f"Wrote {record_count} images")
print(f"Images in output: {len(list((batch_output / 'images').glob('*.png')))}")

# Show first few records
jsonl_path = batch_output / "metadata.jsonl"
with open(jsonl_path) as f:
    lines = f.readlines()
    print(f"\nTotal records: {len(lines)}")
    print("\nFirst 3 records:")
    for line in lines[:3]:
        rec = json.loads(line)
        print(f"  {rec['file']:30s} char='{rec['char']}' font={Path(rec['font_path']).stem}")

## 3. File Naming Convention / ファイル命名規則

File names encode unicode codepoint, font name, and index.
Format: `{unicode_hex}_{font_stem}_{index:03d}.png`

In [ ]:
# Demonstrate naming for different characters
test_chars = ["A", "α", "→"]

naming_output = Path("./test_naming_output")
naming_output.mkdir(exist_ok=True, parents=True)

config = WriterConfig(output_dir=naming_output)
renderer = FontRenderer("./fonts/Aclonica-Regular.ttf")

with DatasetWriter(config) as writer:
    for idx, char in enumerate(test_chars):
        image = renderer.render(char)
        if image:
            filename = writer.write(char, image, "./fonts/Aclonica-Regular.ttf", index=idx)
            codepoint = ord(char)
            print(f"'{char}' (U+{codepoint:04X}) → {filename}")

## 4. Finalization: JSONL → Parquet / ファイナライズ

After writing all records, `finalize()` converts JSONL to Parquet format.

In [ ]:
import pyarrow.parquet as pq

finalize_output = Path("./test_finalize_output")
finalize_output.mkdir(exist_ok=True, parents=True)

# Write some records
config = WriterConfig(output_dir=finalize_output)
renderer = FontRenderer("./fonts/Aclonica-Regular.ttf")

with DatasetWriter(config) as writer:
    for idx, char in enumerate(["A", "B", "C"]):
        image = renderer.render(char)
        if image:
            writer.write(char, image, "./fonts/Aclonica-Regular.ttf", index=idx)

# Finalize (must be done AFTER closing the writer context)
with DatasetWriter(config) as writer:
    pass  # Just open/close

parquet_path = config.output_dir / "metadata.parquet"

# Check both JSONL and Parquet exist
jsonl_path = config.output_dir / "metadata.jsonl"
print(f"JSONL exists: {jsonl_path.exists()}")
print(f"Parquet exists: {parquet_path.exists()}")

# Read Parquet back
if parquet_path.exists():
    table = pq.read_table(str(parquet_path))
    print(f"\nParquet table schema:")
    print(table.schema)
    print(f"\nRecord count: {table.num_rows}")
    print(f"\nFirst record:")
    print(table.to_pylist()[0])

## 5. Context Manager Lifecycle / コンテキストマネージャーのライフサイクル

`DatasetWriter` is a context manager:
- `__enter__` calls `open()` — creates dirs, opens JSONL
- `__exit__` calls `close()` — closes JSONL file
- `finalize()` must be called separately (after all writes complete)

In [ ]:
context_output = Path("./test_context_output")
context_output.mkdir(exist_ok=True, parents=True)

config = WriterConfig(output_dir=context_output)
renderer = FontRenderer("./fonts/Aclonica-Regular.ttf")

print("Before context manager:")
print(f"  Output dir exists: {context_output.exists()}")
print(f"  JSONL exists: {(context_output / 'metadata.jsonl').exists()}")

with DatasetWriter(config) as writer:
    print("\nInside context manager:")
    print(f"  Output dir exists: {(context_output / 'images').exists()}")
    print(f"  JSONL exists: {(context_output / 'metadata.jsonl').exists()}")
    
    image = renderer.render("X")
    writer.write("X", image, "./fonts/Aclonica-Regular.ttf", index=0)

print("\nAfter context manager:")
print(f"  Output dir exists: {context_output.exists()}")
print(f"  JSONL exists: {(context_output / 'metadata.jsonl').exists()}")
print(f"  Images: {list((context_output / 'images').glob('*.png'))}")

## Summary / まとめ

| Task | Method | Output |
|------|--------|--------|
| Single write | `writer.write(char, img, font_path, index)` | PNG + JSONL line |
| Batch write | Loop + context manager | Multiple PNGs + JSONL |
| File naming | `{unicode_hex}_{font_stem}_{index:03d}.png` | Traceable filenames |
| Finalize | `writer.finalize()` | JSONL + Parquet |
| Lifecycle | Context manager | Auto open/close |

**Key design points:**
- Output is **flat**: all images in `images/`, metadata files in root
- JSONL records are written **as-you-go** (streaming)
- Parquet conversion happens **after all writes**
- Writer is **stateless** w.r.t. char/font combinations (pipeline controls index)